In [1]:
import mne
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path 
from scipy.io import loadmat
from mne.decoding import CSP
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.pipeline import Pipeline 

In [2]:
mne.set_log_level('WARNING')

In [3]:
data_folder = Path(r'D:\Coding\BCI project\data')
subjects = ['A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08', 'A09']
ch_renamed = {
    'EEG-Fz':'Fz',
    'EEG-0':'FC3',
    'EEG-1':'FC1',
    'EEG-2':'FCz',
    'EEG-3':'FC2',
    'EEG-4':'FC4',
    'EEG-5':'C5',
    'EEG-C3':'C3',
    'EEG-6':'C1',
    'EEG-Cz':'Cz',
    'EEG-7':'C2',
    'EEG-C4':'C4',
    'EEG-8':'C6',
    'EEG-9':'CP3',
    'EEG-10':'CP1',
    'EEG-11':'CPz',
    'EEG-12':'CP2',
    'EEG-13':'CP4',
    'EEG-14':'P1',
    'EEG-Pz':'Pz',
    'EEG-15':'P2',
    'EEG-16':'POz',
}

In [4]:
def preprocessing(raw, ch_names):
    raw.rename_channels(ch_names)
    raw.set_channel_types({
        'EOG-left':'eog',
        'EOG-central':'eog',
        'EOG-right':'eog'
    })
    montage = mne.channels.make_standard_montage('standard_1005')
    raw.set_montage(montage, on_missing = 'ignore')
    raw.annotations.rename = {'1023':'BAD_1023'}
    raw.filter(l_freq = 1, h_freq = None, fir_design='firwin')
    return raw

In [5]:
def ica_eog(raw):
    ica = mne.preprocessing.ICA(n_components = 20, max_iter = 'auto', random_state = 0)
    ica.fit(raw, reject_by_annotation = True)
    eog_indices, eog_scores = ica.find_bads_eog(raw, threshold = 3.0)
    ica.exclude = eog_indices
    ica.apply(raw)
    return raw

In [6]:
def load_test_labels(subject):
    mat_files = loadmat(Path(r'D:\Coding\BCI project\data/true labels') / f'{subject}E.mat')
    all_test_labels = mat_files['classlabel'].flatten()
    mask = (all_test_labels == 1) | (all_test_labels == 2)
    labels_test = all_test_labels[mask] - 1
    return labels_test, mask

In [7]:
def epoching_train(raw):
    raw.set_eeg_reference(projection = True)
    raw.filter(l_freq=8, h_freq=30, fir_design='firwin')
    events, event_id = mne.events_from_annotations(raw)
    epochs = mne.Epochs(raw, events, 
                        event_id = {'left hand':event_id['769'], 'right hand':event_id['770']}, 
                        tmin = -1, tmax = 4, proj = True, picks = 'eeg', baseline = None, 
                        preload = True)
    epochs_croped = epochs.copy().crop(tmin = 1.0, tmax = 2)
    epochs_data = epochs.get_data(copy = False)
    epochs_croped_data = epochs_croped.get_data(copy = False)
    labels = epochs.events[:, -1] - event_id['769']
    return epochs, epochs_croped, epochs_data, epochs_croped_data, labels

In [8]:
def epoching_test(raw):
    raw.set_eeg_reference(projection = True)
    raw.filter(l_freq=8, h_freq=30, fir_design='firwin')
    events, event_id = mne.events_from_annotations(raw)
    epochs = mne.Epochs(raw, events, event_id = {'unknown':event_id['783']}, tmin = -1,
                        tmax = 4, proj = True, picks = 'eeg', baseline = None, preload = True)
    epochs_croped = epochs.copy().crop(tmin = 1.0, tmax = 2)
    epochs_data = epochs.get_data(copy = False)
    epochs_croped_data = epochs_croped.get_data(copy = False)
    return epochs, epochs_croped, epochs_data, epochs_croped_data

In [9]:
results = {}
class_balance = []

for subject in subjects:
    
    train_files = data_folder / f'{subject}T.gdf' 
    test_files = data_folder / f'{subject}E.gdf'
    raw_train = mne.io.read_raw_gdf(train_files, preload = True)
    raw_test = mne.io.read_raw_gdf(test_files,  preload = True)

    raw_train = preprocessing(raw_train, ch_renamed)
    raw_test = preprocessing(raw_test, ch_renamed)

    raw_train = ica_eog(raw_train)
    raw_test = ica_eog(raw_test)

    labels_test, mask = load_test_labels(subject)

    epochs_train, epochs_train_croped, epochs_train_data, epochs_train_croped_data, labels_train = epoching_train(raw_train)
    epochs_test, epochs_test_croped, epochs_test_data, epochs_test_croped_data = epoching_test(raw_test)
    epochs_test_croped_data = epochs_test_croped_data[mask]

    csp = CSP(n_components = 4, reg = None, log = True, norm_trace = False)
    lda = LinearDiscriminantAnalysis()
    classification = Pipeline([('CSP', csp), ('LDA', lda)])
    classification.fit(epochs_train_croped_data, labels_train)
    score = classification.score(epochs_test_croped_data, labels_test)
    results[subject] = score

    class_balance.append(np.mean(labels_test == labels_test[0]))

final_score = np.mean(list(results.values()))
final_std = np.std(list(results.values()))
class_balance_mean = np.mean(class_balance)
chance_level = max(class_balance_mean, 1 - class_balance_mean)

C:\ProgramData\anaconda3\Lib\contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
C:\ProgramData\anaconda3\Lib\contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
C:\ProgramData\anaconda3\Lib\contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
C:\ProgramData\anaconda3\Lib\contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
C:\ProgramData\anaconda3\Lib\contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
C:\ProgramData\anaconda3\Lib\contextlib.py:148: RuntimeWarning: Channel names are not

In [10]:
print(f'Total classification precision is {final_score * 100}%')
print(f'Standard deviation is {final_std}')
print(f'Chance level is {chance_level * 100}%')
for sbjct, scr in results.items():
    print(f'subject {sbjct} - score {scr}')

Total classification precision is 72.83950617283949%
Standard deviation is 0.15095874318159255
Chance level is 50.0%
subject A01 - score 0.8055555555555556
subject A02 - score 0.5833333333333334
subject A03 - score 0.9166666666666666
subject A04 - score 0.6388888888888888
subject A05 - score 0.5277777777777778
subject A06 - score 0.5625
subject A07 - score 0.7013888888888888
subject A08 - score 0.9236111111111112
subject A09 - score 0.8958333333333334
